# Cookbook: Querying the Multi-Source AI Knowledge Bases

In the previous episodes, we learned how Foundry IQ provides a unified knowledge layer and how different types of content enter the system as Knowledge Sources. Now we'll focus on the **query experience** — how Knowledge Bases act as a single endpoint over multiple sources and how you can control the depth and cost of retrieval.

In this cookbook, you will:

- **Create a multi-source Knowledge Base** with indexed and web Knowledge Sources
- **Query with minimal reasoning effort** — fastest and cheapest, no LLM planning
- **Query with low reasoning effort** — LLM-driven query planning and source selection
- **Query with medium reasoning effort** — iterative retrieval with automatic refinement
- **Compare effort levels side by side** to understand the cost-quality trade-off
- **Explore answer synthesis** — let Foundry IQ generate cited answers vs. returning raw documents
- **Connect via MCP** — use a Knowledge Base as an MCP server endpoint

## Prerequisites

Before running this notebook you need:

1. An **Azure AI Search** service in a [region that supports agentic retrieval](https://learn.microsoft.com/azure/search/search-region-support) with role-based access enabled.
2. A **Microsoft Foundry** project and resource — creating a project automatically provisions the resource.
3. Model deployments: a **text embedding model** (`text-embedding-3-large`) and a **chat model** (`gpt-4o-mini` or equivalent).
4. A **project connection** from your Foundry project to your Azure AI Search service.
5. Appropriate **RBAC roles** on your account: *Search Service Contributor*, *Search Index Data Contributor*, and *Search Index Data Reader* on the search service; *Cognitive Services User* on the Foundry resource for the search service's managed identity.

Create a `.env` file in this directory with your endpoint values:

```
SEARCH_ENDPOINT=https://<your-search-service>.search.windows.net
AOAI_ENDPOINT=https://<your-foundry-resource>.openai.azure.com
AOAI_EMBEDDING_MODEL=text-embedding-3-large
AOAI_EMBEDDING_DEPLOYMENT=text-embedding-3-large
AOAI_GPT_MODEL=gpt-4o-mini
AOAI_GPT_DEPLOYMENT=gpt-4o-mini
```

See the [agentic retrieval quickstart](https://learn.microsoft.com/azure/search/search-get-started-agentic-retrieval) for detailed setup instructions.

## Setup

Install required packages and load environment variables. We authenticate with `DefaultAzureCredential` so no API keys are stored in code.

> **Before you start:** Run `az login` in a terminal to sign in to Azure. This is required for `DefaultAzureCredential` to work.

In [1]:
%%capture
%pip install -U azure-search-documents==11.7.0b2 azure-identity python-dotenv

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

# Service endpoints
SEARCH_ENDPOINT = os.environ["SEARCH_ENDPOINT"]
AOAI_ENDPOINT = os.environ["AOAI_ENDPOINT"]

# Model deployments
EMBEDDING_MODEL = os.environ.get("AOAI_EMBEDDING_MODEL", "text-embedding-3-large")
EMBEDDING_DEPLOYMENT = os.environ.get("AOAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
GPT_MODEL = os.environ.get("AOAI_GPT_MODEL", "gpt-4o-mini")
GPT_DEPLOYMENT = os.environ.get("AOAI_GPT_DEPLOYMENT", "gpt-4o-mini")

# Resource names
INDEX_NAME = "product-catalog-ep3"
INDEXED_SOURCE_NAME = "product-index-source-ep3"
WEB_SOURCE_NAME = "web-source-ep3"
KNOWLEDGE_BASE_NAME = "product-kb-ep3"

credential = DefaultAzureCredential()
print(f"Search endpoint: {SEARCH_ENDPOINT}")
print(f"Models: embedding={EMBEDDING_MODEL}, chat={GPT_MODEL}")

Search endpoint: https://iqs-search-wdxfreotybfvi.search.windows.net
Models: embedding=text-embedding-3-large, chat=gpt-4o-mini


We now have authenticated credentials and all configuration loaded from the environment. Nothing is hard-coded.

---

## Step 1 — Create a Multi-Source Knowledge Base

A Knowledge Base is a single query endpoint that fans out to multiple Knowledge Sources. It has three core responsibilities:

1. **Query planning** — An LLM decomposes the user's question into focused subqueries and selects which sources are relevant.
2. **Knowledge Sources** — Each source is searched independently with the planned subqueries.
3. **Output merging** — Results from all sources are merged, reranked, and optionally synthesized into a final answer.

We'll create a Knowledge Base backed by two sources: an **indexed product catalog** (structured data in a search index) and a **web Knowledge Source** (public web content). This gives us a realistic multi-source setup to explore different query patterns.

First, let's create the search index and upload sample product documents.

In [3]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=[
        SearchField(name="id", type="Edm.String", key=True, filterable=True),
        SearchField(name="title", type="Edm.String", searchable=True),
        SearchField(name="category", type="Edm.String", filterable=True),
        SearchField(name="content", type="Edm.String", searchable=True),
        SearchField(
            name="content_embedding",
            type="Collection(Edm.Single)",
            stored=False,
            vector_search_dimensions=3072,
            vector_search_profile_name="hnsw_text_3_large",
        ),
    ],
    vector_search=VectorSearch(
        profiles=[
            VectorSearchProfile(
                name="hnsw_text_3_large",
                algorithm_configuration_name="alg",
                vectorizer_name="azure_openai_text_3_large",
            )
        ],
        algorithms=[HnswAlgorithmConfiguration(name="alg")],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="azure_openai_text_3_large",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AOAI_ENDPOINT,
                    deployment_name=EMBEDDING_DEPLOYMENT,
                    model_name=EMBEDDING_MODEL,
                ),
            )
        ],
    ),
    # Semantic config is required for agentic retrieval
    semantic_search=SemanticSearch(
        default_configuration_name="semantic_config",
        configurations=[
            SemanticConfiguration(
                name="semantic_config",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="title"),
                    content_fields=[SemanticField(field_name="content")],
                ),
            )
        ],
    ),
)

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
index_client.create_or_update_index(index)
print(f"✅ Index '{INDEX_NAME}' ready")

✅ Index 'product-catalog-ep3' ready


In [4]:
from azure.search.documents import SearchIndexingBufferedSender

documents = [
    {"id": "1", "title": "Zava Premium Interior Paint", "category": "paint", "content": "Zava Premium Interior Paint offers excellent coverage with a smooth, durable finish. Available in matte, eggshell, and semi-gloss finishes. Covers up to 400 sq ft per gallon. Low VOC formula suitable for bedrooms, living rooms, and bathrooms. Price: $38.99 per gallon."},
    {"id": "2", "title": "Zava Exterior WeatherShield Paint", "category": "paint", "content": "Zava Exterior WeatherShield Paint is designed for lasting protection against rain, UV rays, and temperature changes. Available in flat and satin finishes. Self-priming formula covers up to 350 sq ft per gallon. Ideal for wood, stucco, and fiber cement siding. Price: $45.99 per gallon."},
    {"id": "3", "title": "Zava Professional Paint Brush Set", "category": "tools", "content": "This 5-piece brush set includes 1-inch, 2-inch, 3-inch angled, 4-inch flat, and a detail brush. Synthetic bristles designed for use with latex and acrylic paints. Ergonomic handles for comfortable extended use. Price: $24.99 per set."},
    {"id": "4", "title": "Zava Premium Roller Kit", "category": "tools", "content": "Complete roller kit includes a 9-inch roller frame, three roller covers (smooth, semi-smooth, and textured), an extension pole, and a paint tray. Suitable for walls and ceilings. Price: $19.99 per kit."},
    {"id": "5", "title": "Zava Wall Repair Patch Kit", "category": "repair", "content": "All-in-one wall repair kit for patching holes and cracks. Includes spackling compound, putty knife, sanding block, and mesh patches in three sizes. Dries in 30 minutes and can be painted over. Price: $12.99 per kit."},
    {"id": "6", "title": "Zava Bathroom Moisture-Guard Paint", "category": "paint", "content": "Specially formulated for high-humidity environments like bathrooms and kitchens. Mildew-resistant with built-in moisture barrier. Semi-gloss finish for easy cleaning. Covers up to 380 sq ft per gallon. Price: $42.99 per gallon."},
    {"id": "7", "title": "Zava Primer and Sealer", "category": "primer", "content": "Multi-surface primer and sealer works on drywall, wood, and previously painted surfaces. Blocks stains and provides excellent adhesion for topcoats. Quick-dry formula ready for recoat in 1 hour. Price: $29.99 per gallon."},
    {"id": "8", "title": "Zava Color Matching Service", "category": "services", "content": "Bring any color sample to your local Zava store for free computer-assisted color matching. Our spectrophotometer technology ensures 99.5% accuracy. Available for all Zava interior and exterior paint lines. Custom colors mixed in under 5 minutes."},
]

with SearchIndexingBufferedSender(
    endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential
) as sender:
    sender.upload_documents(documents=documents)

print(f"✅ {len(documents)} documents uploaded to '{INDEX_NAME}'")

✅ 8 documents uploaded to 'product-catalog-ep3'


Now let's create the two Knowledge Sources. The **indexed source** points at our product catalog index. The **web source** allows the Knowledge Base to search the public web for supplemental information.

In [5]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    SearchIndexFieldReference,
    WebKnowledgeSource,
)

# Indexed knowledge source — product catalog
indexed_source = SearchIndexKnowledgeSource(
    name=INDEXED_SOURCE_NAME,
    description="Zava product catalog with paints, tools, primers, and services",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="title"),
            SearchIndexFieldReference(name="category"),
        ],
    ),
)

index_client.create_or_update_knowledge_source(knowledge_source=indexed_source)
print(f"✅ Indexed knowledge source '{INDEXED_SOURCE_NAME}' ready")

# Web knowledge source — public web
web_source = WebKnowledgeSource(
    name=WEB_SOURCE_NAME,
    description="Public web for general painting tips, techniques, and product reviews",
)

index_client.create_or_update_knowledge_source(knowledge_source=web_source)
print(f"✅ Web knowledge source '{WEB_SOURCE_NAME}' ready")

✅ Indexed knowledge source 'product-index-source-ep3' ready
✅ Web knowledge source 'web-source-ep3' ready


Finally, we create the Knowledge Base that ties both sources together behind a single endpoint. We use `ANSWER_SYNTHESIS` mode so the Knowledge Base generates natural-language answers with inline citations — this is required when a Web Knowledge Source is included.

In [6]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
    KnowledgeRetrievalOutputMode,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    models=[
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AOAI_ENDPOINT,
                deployment_name=GPT_DEPLOYMENT,
                model_name=GPT_MODEL,
            )
        )
    ],
    knowledge_sources=[
        KnowledgeSourceReference(name=INDEXED_SOURCE_NAME),
        KnowledgeSourceReference(name=WEB_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Provide a helpful, concise answer grounded in the retrieved documents. Include product names and prices when available.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"✅ Knowledge base '{KNOWLEDGE_BASE_NAME}' ready with 2 sources")

✅ Knowledge base 'product-kb-ep3' ready with 2 sources


We now have a Knowledge Base with two sources: an indexed product catalog and the public web. This gives us a realistic multi-source setup to explore different query patterns.

---

## Step 2 — Query with Minimal Reasoning Effort

**Minimal effort** is the fastest and cheapest option because it doesn't use an LLM at all during retrieval. You provide the search intents directly — your agent or application has already decided what to search for.

With minimal effort:
- You provide explicit `search_intents` instead of a conversation
- Every intent is sent to every Knowledge Source
- Results are merged and reranked
- No LLM tokens are consumed during retrieval

This works best when integrating with an agent that already does its own reasoning — like the Foundry Agent Service — so you avoid paying for LLM processing twice.

In [10]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalSemanticIntent,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalMediumReasoningEffort,
)
import json

retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=SEARCH_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=credential,
)

# With minimal effort, use intents (not messages)
result_minimal = retrieval_client.retrieve(
    retrieval_request=KnowledgeBaseRetrievalRequest(
        intents=[
            KnowledgeRetrievalSemanticIntent(
                search="What Zava paint is best for bathrooms?"
            )
        ],
        include_activity=True,
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    )
)

print("✅ Minimal effort retrieval complete")

✅ Minimal effort retrieval complete


In [12]:
if result_minimal.activity:
    print("🔍 Activity log (minimal effort):")
    print(json.dumps([a.as_dict() for a in result_minimal.activity], indent=2))

if result_minimal.references:
    print(f"\n📚 {len(result_minimal.references)} reference(s) returned")

🔍 Activity log (minimal effort):
[
  {
    "id": 0,
    "type": "searchIndex",
    "elapsed_ms": 266,
    "knowledge_source_name": "product-index-source-ep3",
    "query_time": "2026-05-03T07:51:34.128Z",
    "count": 7,
    "search_index_arguments": {
      "search": "What Zava paint is best for bathrooms?",
      "source_data_fields": [
        {
          "name": "title"
        },
        {
          "name": "content"
        },
        {
          "name": "id"
        },
        {
          "name": "category"
        }
      ],
      "search_fields": [],
      "semantic_configuration_name": "semantic_config"
    }
  },
  {
    "id": 1,
    "type": "web",
    "elapsed_ms": 2358,
    "knowledge_source_name": "web-source-ep3",
    "query_time": "2026-05-03T07:51:34.128Z",
    "count": 9,
    "web_arguments": {
      "search": "What Zava paint is best for bathrooms?"
    }
  },
  {
    "id": 2,
    "type": "agenticReasoning",
    "reasoning_tokens": 14937,
    "retrieval_reasoning_eff

Notice how the activity log shows a straightforward path: the query goes directly to all sources, results are merged and reranked. There's no query planning step because minimal effort skips LLM processing entirely. This makes it the fastest option — typically completing in a fraction of the time of higher effort levels.

---

## Step 3 — Query with Low Reasoning Effort

**Low effort** is the default, and it's where you start seeing the power of agentic retrieval. The Knowledge Base uses an LLM for **query planning**:

- The LLM analyzes the full conversation to generate focused subqueries
- It selects which Knowledge Sources are relevant (so irrelevant sources aren't searched)
- Subqueries fan out to selected sources in parallel
- Results are merged and reranked

For example, if you ask "What Zava paint is best for bathrooms and how much does it cost?" — that's really two information needs. The LLM breaks it down into separate subqueries, each targeting what it needs.

In [13]:
query = "Does Zava have exterior paint and how much does it cost?"

result_low = retrieval_client.retrieve(
    retrieval_request=KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role="user",
                content=[KnowledgeBaseMessageTextContent(text=query)],
            )
        ],
        include_activity=True,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    )
)

print("✅ Low effort retrieval complete")

✅ Low effort retrieval complete


In [14]:
if result_low.activity:
    print("🔍 Activity log (low effort):")
    print(json.dumps([a.as_dict() for a in result_low.activity], indent=2))

if result_low.references:
    print(f"\n📚 {len(result_low.references)} reference(s) returned")

🔍 Activity log (low effort):
[
  {
    "id": 0,
    "type": "modelQueryPlanning",
    "elapsed_ms": 1159,
    "input_tokens": 2125,
    "output_tokens": 88
  },
  {
    "id": 1,
    "type": "searchIndex",
    "elapsed_ms": 166,
    "knowledge_source_name": "product-index-source-ep3",
    "query_time": "2026-05-03T07:52:37.686Z",
    "count": 3,
    "search_index_arguments": {
      "search": "Zava exterior paint availability",
      "source_data_fields": [
        {
          "name": "title"
        },
        {
          "name": "content"
        },
        {
          "name": "id"
        },
        {
          "name": "category"
        }
      ],
      "search_fields": [],
      "semantic_configuration_name": "semantic_config"
    }
  },
  {
    "id": 2,
    "type": "searchIndex",
    "elapsed_ms": 128,
    "knowledge_source_name": "product-index-source-ep3",
    "query_time": "2026-05-03T07:52:37.814Z",
    "count": 7,
    "search_index_arguments": {
      "search": "Zava exterior

Compare this to minimal effort — the activity log now starts with an LLM query planning phase. The LLM decomposed the question into focused subqueries and selected which sources are relevant. Notice that it may skip certain sources that aren't relevant to the question, saving unnecessary queries and reducing noise in the results.

---

## Step 4 — Query with Medium Reasoning Effort

**Medium effort** is for when you need the most comprehensive results. It adds **iterative retrieval** — after the first search, the system evaluates whether it has enough information:

1. **First pass** — Query planning and parallel search (like low effort)
2. **Evaluation** — A semantic classifier asks: "Is there enough information to answer? Did we find highly relevant documents?"
3. **Second pass** (if needed) — Refined queries based on what was found and what's missing

This is similar to how a human researcher works: search, evaluate, realize something is missing, and refine your approach. The second iteration is smarter because it has context from the first pass.

Let's try a complex question that benefits from iterative retrieval.

In [15]:
query = "Explain how to paint my house most efficiently, then give me a list of Zava products and prices for each supply I would need."

result_medium = retrieval_client.retrieve(
    retrieval_request=KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role="user",
                content=[KnowledgeBaseMessageTextContent(text=query)],
            )
        ],
        include_activity=True,
        retrieval_reasoning_effort=KnowledgeRetrievalMediumReasoningEffort(),
    )
)

print("✅ Medium effort retrieval complete")

✅ Medium effort retrieval complete


In [16]:
if result_medium.activity:
    print("🔍 Activity log (medium effort):")
    print(json.dumps([a.as_dict() for a in result_medium.activity], indent=2))

if result_medium.references:
    print(f"\n📚 {len(result_medium.references)} reference(s) returned")

🔍 Activity log (medium effort):
[
  {
    "id": 0,
    "type": "modelQueryPlanning",
    "elapsed_ms": 1177,
    "input_tokens": 2139,
    "output_tokens": 126
  },
  {
    "id": 1,
    "type": "searchIndex",
    "elapsed_ms": 170,
    "knowledge_source_name": "product-index-source-ep3",
    "query_time": "2026-05-03T07:53:15.302Z",
    "count": 3,
    "search_index_arguments": {
      "search": "how to paint a house efficiently",
      "source_data_fields": [
        {
          "name": "title"
        },
        {
          "name": "content"
        },
        {
          "name": "id"
        },
        {
          "name": "category"
        }
      ],
      "search_fields": [],
      "semantic_configuration_name": "semantic_config"
    }
  },
  {
    "id": 2,
    "type": "searchIndex",
    "elapsed_ms": 117,
    "knowledge_source_name": "product-index-source-ep3",
    "query_time": "2026-05-03T07:53:15.419Z",
    "count": 8,
    "search_index_arguments": {
      "search": "Zava prod

The activity log for medium effort tells a richer story. You can see the first pass of queries, the evaluation step where the semantic classifier determines if more information is needed, and potentially a second pass with refined queries. This iterative process consumes more tokens but produces more comprehensive results — ideal for complex, multi-part questions.

---

## Step 5 — Compare Effort Levels Side by Side

Now let's run the same query at all three effort levels and compare the results. This helps visualize the trade-off between speed/cost and result quality.

In [18]:
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeRetrievalSemanticIntent,
)

comparison_query = "What Zava products do I need to repaint a bathroom, and what will it cost?"

results = {}
for effort_name, effort_level in [
    ("minimal", KnowledgeRetrievalMinimalReasoningEffort()),
    ("low", KnowledgeRetrievalLowReasoningEffort()),
    ("medium", KnowledgeRetrievalMediumReasoningEffort()),
]:
    if effort_name == "minimal":
        request = KnowledgeBaseRetrievalRequest(
            intents=[KnowledgeRetrievalSemanticIntent(search=comparison_query)],
            include_activity=True,
            retrieval_reasoning_effort=effort_level,
        )
    else:
        request = KnowledgeBaseRetrievalRequest(
            messages=[
                KnowledgeBaseMessage(
                    role="user",
                    content=[KnowledgeBaseMessageTextContent(text=comparison_query)],
                )
            ],
            include_activity=True,
            retrieval_reasoning_effort=effort_level,
        )

    result = retrieval_client.retrieve(retrieval_request=request)
    results[effort_name] = result

print("✅ All three effort levels retrieved")

✅ All three effort levels retrieved


In [19]:
print(f"{'Effort Level':<15} {'Activity Steps':<18} {'References':<12}")
print("-" * 45)
for name, result in results.items():
    activity_count = len(result.activity) if result.activity else 0
    ref_count = len(result.references) if result.references else 0
    print(f"{name:<15} {activity_count:<18} {ref_count:<12}")

Effort Level    Activity Steps     References  
---------------------------------------------
minimal         4                  11          
low             6                  8           
medium          18                 14          


This comparison shows the trade-off clearly:

- **Minimal** — Fewest steps, fastest execution, least expensive. Use when your agent already plans queries.
- **Low** — Adds LLM planning for smarter query decomposition and source selection. Good default for most scenarios.
- **Medium** — Adds iterative refinement for the most comprehensive results. Use for complex questions where coverage matters more than speed.

Choose the right level per scenario — there's no single best answer.

---

## Step 6 — Answer Synthesis

Our Knowledge Base is already using `ANSWER_SYNTHESIS` mode (required when a Web Knowledge Source is included). Let’s explore what this means and how to customize the synthesized answers.

There are two output modes in Foundry IQ:
- `EXTRACTIVE_DATA` — Returns raw document chunks and references for your agent to process (only available for indexed-only Knowledge Bases)
- `ANSWER_SYNTHESIS` — The Knowledge Base generates a natural-language answer with inline citations

Answer synthesis is especially useful when the response needs to combine information from multiple sources and include correct inline citations — a tricky task for downstream agents to get right consistently.

Let’s update the `answer_instructions` to guide the synthesis output.

In [20]:
# Update the Knowledge Base with refined answer instructions
knowledge_base_with_synthesis = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    models=[
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AOAI_ENDPOINT,
                deployment_name=GPT_DEPLOYMENT,
                model_name=GPT_MODEL,
            )
        )
    ],
    knowledge_sources=[
        KnowledgeSourceReference(name=INDEXED_SOURCE_NAME),
        KnowledgeSourceReference(name=WEB_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Provide a helpful, concise answer grounded in the retrieved documents. Include product names and prices when available.",
)

index_client.create_or_update_knowledge_base(knowledge_base_with_synthesis)
print("✅ Knowledge Base updated with answer synthesis")

✅ Knowledge Base updated with answer synthesis


In [21]:
result_synthesis = retrieval_client.retrieve(
    retrieval_request=KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role="user",
                content=[KnowledgeBaseMessageTextContent(text="What Zava paint should I use in my bathroom and how much does it cost?")],
            )
        ],
        include_activity=True,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    )
)

# Extract the synthesized answer
answer = "\n\n".join(
    content.text
    for resp in result_synthesis.response
    for content in resp.content
)
print("📝 Synthesized Answer:\n")
print(answer)

if result_synthesis.references:
    print(f"\n📚 {len(result_synthesis.references)} reference(s) grounded the answer")

📝 Synthesized Answer:

For your bathroom, you should consider using Zava Bathroom Moisture-Guard Paint, which is specially formulated for high-humidity environments. It is mildew-resistant and has a built-in moisture barrier, making it ideal for bathrooms. The paint has a semi-gloss finish for easy cleaning and covers up to 380 sq ft per gallon. The price is $42.99 per gallon [ref_id:0]. 

Additionally, Zava Premium Interior Paint is another option, offering excellent coverage with a smooth, durable finish. It is available in matte, eggshell, and semi-gloss finishes, covering up to 400 sq ft per gallon, and costs $38.99 per gallon [ref_id:3]. 

Both options are suitable for bathroom use, but the Bathroom Moisture-Guard Paint is specifically designed for such environments.

📚 32 reference(s) grounded the answer


With answer synthesis, the Knowledge Base does the heavy lifting of combining information from multiple sources into a coherent, cited answer. This removes the burden from your agent and ensures consistent citation formatting.

---

## Step 7 — Connect via MCP

Every Knowledge Base in Foundry IQ exposes an **MCP (Model Context Protocol) endpoint** that any MCP-compatible client can connect to. This means you can use your Knowledge Base from GitHub Copilot, Claude Desktop, VS Code, or any other MCP client — without writing any retrieval code.

The MCP endpoint follows this pattern:

```
https://<your-search-service>.search.windows.net/knowledgebases/<your-kb-name>/mcp?api-version=2025-11-01-preview
```

### GitHub Copilot CLI

Edit `~/.copilot/mcp-config.json`:

```json
{
  "mcpServers": {
    "my-kb": {
      "url": "https://<your-search-service>.search.windows.net/knowledgebases/<your-kb-name>/mcp?api-version=2025-11-01-preview",
      "headers": {
        "api-key": "<your-query-key>"
      }
    }
  }
}
```

### GitHub Copilot in VS Code

Create `.vscode/mcp.json` in your workspace:

```json
{
  "servers": {
    "my-kb": {
      "type": "sse",
      "url": "https://<your-search-service>.search.windows.net/knowledgebases/<your-kb-name>/mcp?api-version=2025-11-01-preview",
      "headers": {
        "api-key": "<your-query-key>"
      }
    }
  }
}
```

### Claude Desktop

Edit `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "my-kb": {
      "command": "npx",
      "args": [
        "mcp-remote",
        "https://<your-search-service>.search.windows.net/knowledgebases/<your-kb-name>/mcp?api-version=2025-11-01-preview",
        "--header", "api-key: <your-query-key>"
      ]
    }
  }
}
```

The MCP client sends queries as intents, the Knowledge Base handles multi-source search behind the scenes, and results flow back for the client to synthesize. The client doesn't need to know anything about how Foundry IQ routes queries — it just sends intents and reasons over the merged results.

> **Tip:** Replace `<your-search-service>`, `<your-kb-name>`, and `<your-query-key>` with your actual values. For Knowledge Bases with remote SharePoint sources, include the `x-ms-query-source-authorization` header with a bearer token.

---

## Clean Up

Remove the Foundry IQ resources we created. If you want to keep experimenting, skip this cell.

In [ ]:
index_client.delete_knowledge_base(KNOWLEDGE_BASE_NAME)
print(f"Deleted knowledge base '{KNOWLEDGE_BASE_NAME}'")

for source_name in [INDEXED_SOURCE_NAME, WEB_SOURCE_NAME]:
    try:
        index_client.delete_knowledge_source(knowledge_source=source_name)
        print(f"Deleted knowledge source '{source_name}'")
    except Exception:
        pass

index_client.delete_index(INDEX_NAME)
print(f"Deleted index '{INDEX_NAME}'")

---

## Conclusion

In this cookbook we explored how to query multi-source Knowledge Bases effectively:

1. **Multi-source Knowledge Base** — We created a Knowledge Base with an indexed product catalog and web sources behind a single endpoint.
2. **Minimal effort** — No LLM processing during retrieval. Fastest and cheapest, best when your agent already plans queries.
3. **Low effort** — LLM-driven query planning decomposes questions and selects relevant sources. Good default for most conversational scenarios.
4. **Medium effort** — Iterative retrieval evaluates result quality and refines queries automatically. Best for complex, multi-part questions where comprehensive coverage matters.
5. **Answer synthesis** — Let the Knowledge Base generate cited answers instead of returning raw documents.
6. **MCP integration** — Any MCP-compatible client can connect to a Knowledge Base endpoint.

### Where to go next

- **Tune source descriptions** — Better descriptions lead to smarter source selection during query planning.
- **Experiment with answer instructions** — Guide the synthesis LLM to produce answers tailored to your use case.
- **Monitor with activity logs** — Use the activity log for debugging retrieval quality and monitoring token usage in production.
- **Explore the IQ Series** — Review [Episode 1: Unified Knowledge Layer](../../1-Foundry-IQ-Unified-Knowledge-Layer-for-Agents/) and [Episode 2: Building the Data Pipeline](../../2-Foundry-IQ-Building-the-Data-Pipeline-with-Knowledge-Sources/) for the full picture.